# Order Performance — POC

**Spec:** `spec.md` v0.6  
**Scope:** Jan–Feb 2026 · US  
**Date:** 2026-03-14

> **Nguyên tắc:** Notebook này chỉ visualize — không transform, filter, hay tính toán thêm.  
> Toàn bộ logic đã xử lý trong SQL. Data load từ CSV exports của MCP.
>
> | File | Query | Nội dung |
> |---|---|---|
> | `queries/kpi.csv` | `kpi.sql` | Overall KPI snapshot |
> | `queries/segment.csv` | `segment.sql` | Breakdown by facility × shipping_class |
> | `queries/cdf.csv` | `cdf.sql` | Distribution breakpoints cho 3 metrics |
>
> Static PNG export: chạy `python3 export.py`

## 1. Setup

In [ ]:
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

SCOPE = 'Jan–Feb 2026 · US'
C_PROD, C_LAG, C_P50, C_P90 = '#2ca02c', '#d62728', '#1f77b4', '#ff7f0e'
pct_fmt = lambda v: f'{v:.1f}%'

## 2. Load Data

In [ ]:
kpi = pd.read_csv('queries/kpi.csv').iloc[0]
seg = pd.read_csv('queries/segment.csv')
cdf = pd.read_csv('queries/cdf.csv').iloc[0]

n_orders = int(kpi['orders'])
print(f'KPI: {n_orders:,} orders | Segments: {len(seg)} | Scope: {SCOPE}')
display(seg)

## 3. Chart 1 — KPI Summary

In [ ]:
fig = go.Figure()

metrics = [
    ('P50 Active Lead', kpi['p50_active_lead_d'], 'days', C_P50),
    ('P90 Active Lead', kpi['p90_active_lead_d'], 'days', C_P90),
    ('Avg Active Lead', kpi['avg_active_lead_d'], 'days', '#7f7f7f'),
    ('Same-day Scan Rate', kpi['pct_scan_within_1d'], '%', C_PROD),
]

for i, (label, value, unit, color) in enumerate(metrics):
    fig.add_annotation(
        x=i / (len(metrics) - 1), y=0.6, xref='paper', yref='paper',
        text=f'<b style="font-size:28px;color:{color}">{value}{unit}</b>',
        showarrow=False
    )
    fig.add_annotation(
        x=i / (len(metrics) - 1), y=0.3, xref='paper', yref='paper',
        text=f'<span style="color:#555">{label}</span>',
        showarrow=False
    )

fig.update_layout(
    title=f'KPI Summary — Tracking Active Lead  ({SCOPE}, n={n_orders:,})',
    height=250, xaxis_visible=False, yaxis_visible=False,
    plot_bgcolor='#f8f9fa', paper_bgcolor='#f8f9fa',
)
fig.show()

## 4. Chart 2 — Lead Time Breakdown: Production vs Tracking Lag

In [ ]:
total = kpi['avg_prod_d'] + kpi['avg_lag_d']
pct_p = kpi['avg_prod_d'] / total * 100
pct_l = kpi['avg_lag_d']  / total * 100

fig = go.Figure()
fig.add_trace(go.Bar(
    x=[kpi['avg_prod_d']], y=[''], orientation='h', name=f"Production  {kpi['avg_prod_d']:.2f}d ({pct_p:.0f}%)",
    marker_color=C_PROD,
    text=f"{kpi['avg_prod_d']:.2f}d ({pct_p:.0f}%)", textposition='inside', textfont_color='white', textfont_size=14,
))
fig.add_trace(go.Bar(
    x=[kpi['avg_lag_d']], y=[''], orientation='h', name=f"Tracking Lag  {kpi['avg_lag_d']:.2f}d ({pct_l:.0f}%)",
    marker_color=C_LAG,
    text=f"{kpi['avg_lag_d']:.2f}d ({pct_l:.0f}%)", textposition='inside', textfont_color='white', textfont_size=14,
))
fig.update_layout(
    barmode='stack',
    title=f'Lead Time Breakdown — Production vs Carrier  ({SCOPE})',
    xaxis_title='Days (avg)', height=200,
    legend=dict(orientation='h', yanchor='bottom', y=1.1, xanchor='right', x=1),
)
fig.show()

## 5. Chart 3/4/4b — Distribution Curves

In [ ]:
def plot_cdf_plotly(points, title, xlabel):
    x_vals = [p[0] for p in points]
    y_vals = [float(cdf[col]) for _, col in points]
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=x_vals, y=y_vals, mode='lines+markers+text',
        line=dict(color=C_P50, width=2.5), marker=dict(size=8),
        text=[f'{y:.1f}%' for y in y_vals], textposition='top center',
    ))
    fig.update_layout(
        title=f'{title}<br><sup>{SCOPE}  ·  n={n_orders:,}</sup>',
        xaxis_title=xlabel, yaxis_title='% Orders (cumulative)',
        yaxis=dict(ticksuffix='%', range=[0, 115]),
        xaxis=dict(tickvals=x_vals), height=400,
    )
    fig.show()

plot_cdf_plotly(
    [(0.5,'prod_le_0d5'),(1,'prod_le_1d'),(2,'prod_le_2d'),(3,'prod_le_3d'),(5,'prod_le_5d'),(7,'prod_le_7d')],
    'Production Time Distribution — % Orders Fulfilled Within X Days',
    'Production Time (days)'
)
plot_cdf_plotly(
    [(0.25,'lag_le_0d25'),(0.5,'lag_le_0d5'),(1,'lag_le_1d'),(2,'lag_le_2d'),(3,'lag_le_3d')],
    'Tracking Lag Distribution — % Orders First-Scanned Within X Biz Days',
    'Tracking Lag (business days)'
)
plot_cdf_plotly(
    [(1,'lead_le_1d'),(2,'lead_le_2d'),(3,'lead_le_3d'),(5,'lead_le_5d'),(7,'lead_le_7d')],
    'Tracking Active Lead Distribution — % Orders Completed Within X Days',
    'Tracking Active Lead (days)'
)

## 6. Chart 5 — P50 Production Time by Facility × Shipping Class

In [ ]:
fig = px.bar(
    seg, x='facility', y='p50_prod_d', color='shipping_class',
    barmode='group', text='p50_prod_d',
    title=f'P50 Production Time — Facility × Shipping Class  ({SCOPE})',
    labels={'facility': 'Facility', 'p50_prod_d': 'Days (P50)', 'shipping_class': 'Shipping Class'},
)
fig.update_traces(texttemplate='%{text:.2f}', textposition='outside')
fig.update_layout(height=420, yaxis_title='Days (P50)')
fig.show()

## 7. Chart 6 — Same-day Scan Rate by Facility × Shipping Class

In [ ]:
seg['label'] = seg['facility'] + ' · ' + seg['shipping_class']

fig = px.bar(
    seg, x='label', y='pct_scan_within_1d',
    color='pct_scan_within_1d', color_continuous_scale='RdYlGn',
    text='pct_scan_within_1d',
    title=f'Same-day Scan Rate (≤ 1 biz day) — by Segment  ({SCOPE})',
    labels={'label': '', 'pct_scan_within_1d': '% Orders'},
    hover_data={'orders': True},
)
fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.update_layout(
    height=420, yaxis=dict(ticksuffix='%', range=[0, 115]),
    coloraxis_showscale=False,
)
fig.show()

## 8. Notes

- **Findings:** (điền sau khi review charts)
- **Edge cases:** Segment HB2 Standard chỉ có 113 orders — interpret với cẩn thận.
- **SQL cần điều chỉnh:** Không.
- **Next steps (Metabase):** Upload 4 queries, tạo dashboard với 7 charts theo spec.